# 05y feature approval and dictionary patch2 260515

Purpose: patch semantic review issues in the previous 05y feature approval and feature dictionary package before 06x dataset generation.

Scope guard: contract and dictionary quality patch only. No modeling, EDA, SHAP, Optuna, or segmentation is performed.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import re
import zipfile

import numpy as np
import pandas as pd

STEP = '05y_feature_approval_and_dictionary_patch2_260515'
START_CWD = Path.cwd().resolve()
repo_candidates = [p for p in [START_CWD, *START_CWD.parents] if (p / '.git').exists() and (p / 'park.ingyeom').exists()]
if repo_candidates:
    REPO_ROOT = repo_candidates[0].resolve()
else:
    park_candidates = [p for p in [START_CWD, *START_CWD.parents] if p.name == 'park.ingyeom']
    if not park_candidates:
        raise RuntimeError(f'Cannot resolve repo root from cwd={START_CWD}')
    REPO_ROOT = park_candidates[0].parent.resolve()
PARK = REPO_ROOT / 'park.ingyeom'
DATA = PARK / 'data'
NOTEBOOK_DIR = PARK / 'notebook' / STEP
OUT_DIR = PARK / 'reports' / 'audits' / STEP
ZIP_DIR = PARK / 'zip'
NOTE_PATH = PARK / 'note.md'
ZIP_PATH = ZIP_DIR / f'{STEP}_review_package.zip'
NOTEBOOK_PATH = NOTEBOOK_DIR / f'{STEP}.ipynb'

SOURCE_MASTER = DATA / '(광일)Membership_v2_with_derived_features.csv'
V3_CSV = DATA / '변수_합집합_비교_v3.csv'
PREV_05Y = PARK / 'reports' / 'audits' / '05y_feature_approval_and_dictionary_260515'
PATCH_05X = PARK / 'reports' / 'audits' / '05x_feature_contract_rebuild_patch_260515'
RAW_FILES = {
    'View_History_v2.csv': DATA / 'View_History_v2.csv',
    'User_Mapping_v2.csv': DATA / 'User_Mapping_v2.csv',
    'Membership_train.csv': DATA / 'Membership_train.csv',
    'Movie_Master_v2.csv': DATA / 'Movie_Master_v2.csv',
}

NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

def inside(child, parent):
    child = Path(child).resolve()
    parent = Path(parent).resolve()
    return child == parent or parent in child.parents

assert inside(OUT_DIR, PARK)
assert inside(NOTEBOOK_DIR, PARK)
assert inside(ZIP_PATH, PARK)
assert not inside(OUT_DIR, REPO_ROOT / '_data')
assert not inside(OUT_DIR, REPO_ROOT / '.tmp')

def read_csv_any(path, **kwargs):
    last_error = None
    for enc in ['utf-8-sig', 'utf-8', 'cp949']:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error

raw_paths = [SOURCE_MASTER, V3_CSV, *RAW_FILES.values()]
raw_stats_before = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in raw_paths if p.exists()}

preflight_rows = []
def add_preflight(item, path=None, exists=None, status=None, stop_reason=''):
    if exists is None and path is not None:
        exists = Path(path).exists()
    preflight_rows.append({
        'item': item,
        'path': str(path) if path is not None else '',
        'exists': bool(exists),
        'status': status or ('PASS' if exists else 'FAIL'),
        'stop_reason': stop_reason,
    })

add_preflight('source master exists', SOURCE_MASTER)
add_preflight('v3 csv exists', V3_CSV)
add_preflight('previous 05y folder exists', PREV_05Y)
add_preflight('05x patch folder exists', PATCH_05X)
for name, path in RAW_FILES.items():
    add_preflight(f'raw validation file exists: {name}', path)
add_preflight('output folder', OUT_DIR, exists=OUT_DIR.exists(), status='PASS')
preflight = pd.DataFrame(preflight_rows)
if (preflight['status'] == 'FAIL').any():
    preflight.loc[preflight['status'] == 'FAIL', 'stop_reason'] = 'required input missing'
    preflight.to_csv(OUT_DIR / '05y_patch2_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
    raise FileNotFoundError(preflight[preflight['status'] == 'FAIL'].to_string(index=False))

master = read_csv_any(SOURCE_MASTER)
v3 = read_csv_any(V3_CSV)
prev_decision = read_csv_any(PREV_05Y / '05y_user_approval_decision_table.csv')
prev_conservative = read_csv_any(PREV_05Y / '05y_conservative_safe_feature_contract.csv')
prev_expanded = read_csv_any(PREV_05Y / '05y_expanded_feature_contract.csv')
prev_excluded = read_csv_any(PREV_05Y / '05y_excluded_feature_contract.csv')

master_cols = list(master.columns)
master_col_set = set(master_cols)
generated_features = ['is_basic', 'is_cold_start_3d_fixed', 'is_cold_start_7d_fixed']

def safe_name(name):
    s = str(name).strip()
    s = s.replace('(min)', '_min')
    s = s.replace('(5y)', '_5y')
    s = s.replace('%', 'pct')
    s = re.sub(r'[^0-9A-Za-z가-힣_]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    if re.match(r'^\d', s):
        s = 'feature_' + s
    return s

excluded_user = {
    'product_code', 'billing_method', 'payment_device', 'gender', 'age', 'reg_hour',
    'price', 'max_screen', 'reg_date', 'end_date', 'USER_KEY', 'is_repurchase',
    'is_cold_start_3d', 'is_cold_start_7d',
}
audit_only = {'USER_KEY', 'reg_date', 'end_date'}
target = {'is_repurchase'}
special_limited = {'is_promotion'}
approved_generated = set(generated_features)

all_features = master_cols + [f for f in generated_features if f not in master_col_set]
mapping = pd.DataFrame({'original_feature_name': all_features})
mapping['safe_model_feature_name'] = mapping['original_feature_name'].map(safe_name)
mapping['rename_rule_applied'] = np.where(
    mapping['original_feature_name'] == mapping['safe_model_feature_name'],
    'none',
    'non_alphanumeric_parenthesis_percent_or_space_normalized',
)
dup_safe = mapping['safe_model_feature_name'].duplicated(keep=False)
mapping['collision_check'] = np.where(dup_safe, 'collision', 'unique')
mapping['status'] = np.where(dup_safe, 'FAIL', 'PASS')

def family(feature):
    if feature in {'is_standard', 'is_premium', 'is_basic', 'is_churn_prevented', 'is_promotion', 'is_user_verified'}:
        return 'membership_context'
    if feature in {'age_group', 'is_female', 'is_male'}:
        return 'demographic_derived'
    if feature.startswith('reg_'):
        return 'registration_time_derived'
    if feature.startswith('payment_'):
        return 'payment_device_derived'
    if feature.startswith('new_movie') or feature.startswith('old_movie') or feature == 'avg_ott_release_year':
        return 'content_recency'
    if 'ratio' in feature or feature == 'genre_diversity_count':
        if feature in {'action_adventure_ratio', 'family_animation_ratio', 'drama_ratio', 'thriller_crime_ratio', 'sf_fantasy_ratio', 'comedy_ratio', 'romance_ratio', 'horror_ratio', 'documentary_ratio', 'historical_war_ratio', 'other_ratio', 'genre_diversity_count'}:
            return 'genre_content'
    if feature in generated_features:
        return 'fixed_policy_generated'
    return 'usage_summary'

def source_cols(feature):
    explicit = {
        'USER_KEY': 'membership/source master USER_KEY',
        'product_code': 'membership/source master product_code',
        'billing_method': 'membership/source master billing_method',
        'price': 'membership/source master price',
        'max_screen': 'membership/source master max_screen',
        'gender': 'membership/source master gender',
        'age': 'membership/source master age',
        'reg_hour': 'membership/source master reg_hour',
        'reg_date': 'membership/source master reg_date',
        'end_date': 'membership/source master end_date',
        'is_repurchase': 'source master is_repurchase target label',
        'is_basic': 'max_screen',
        'is_standard': 'max_screen',
        'is_premium': 'max_screen',
        'age_group': 'age',
        'is_female': 'gender',
        'is_male': 'gender',
        'reg_is_weekend': 'reg_date',
        'reg_hour_morning': 'reg_hour',
        'reg_hour_afternoon': 'reg_hour',
        'reg_hour_evening': 'reg_hour',
        'reg_hour_night': 'reg_hour',
        'payment_is_mobile': 'payment_device',
        'payment_is_pc': 'payment_device',
        'payment_is_android': 'payment_device',
        'payment_is_ios': 'payment_device',
        'is_cold_start_3d_fixed': 'reg_date, View_History_v2.watch_day, User_Mapping_v2.USER_NUM',
        'is_cold_start_7d_fixed': 'reg_date, View_History_v2.watch_day, User_Mapping_v2.USER_NUM',
        'is_cold_start_3d': 'source master is_cold_start_3d',
        'is_cold_start_7d': 'source master is_cold_start_7d',
        'old_movie_ratio(5y)': 'source master old_movie_ratio(5y); Movie_Master_v2 used for caveat only',
    }
    if feature in explicit:
        return explicit[feature]
    if feature in {'is_churn_prevented', 'is_promotion', 'is_user_verified'}:
        return f'Membership_v2.{feature}'
    if family(feature) in {'content_recency', 'genre_content'}:
        return 'View_History_v2, Movie_Master_v2, User_Mapping_v2'
    if family(feature) == 'usage_summary':
        return 'View_History_v2 grouped by USER_NUM/USER_KEY within observation window'
    return 'source master'

formula_map = {
    'USER_KEY': 'identifier / group key only; not a model feature',
    'product_code': 'source membership product code; excluded by user approval',
    'price': 'source membership price; excluded by user approval',
    'billing_method': 'source billing method; excluded by user approval',
    'max_screen': 'source concurrent screen count; excluded after derived screen-plan flags are used',
    'payment_device': 'source payment device; excluded after payment_is_* flags are used',
    'gender': 'source gender; excluded after is_male/is_female flags are used',
    'age': 'source age; excluded after age_group is used',
    'reg_date': 'source registration date; audit/calculation reference only, not a model feature',
    'reg_hour': 'source registration hour; excluded after time-band flags are used',
    'end_date': 'source subscription end date; audit/policy reference only, not a model feature',
    'is_repurchase': 'target variable; not a model feature',
    'is_standard': '1 if max_screen == 2 else 0',
    'is_premium': '1 if max_screen == 4 else 0',
    'is_basic': '1 if max_screen is neither 2 nor 4 else 0; generated in 06x from max_screen',
    'is_churn_prevented': 'source master value used as-is; historical ever received churn-prevention benefit flag',
    'is_promotion': 'source master value used as-is; split key and only allowed as a feature in overall_with_promotion',
    'is_user_verified': 'source master value used as-is; membership/user verification context flag',
    'age_group': 'floor(age / 10) * 10 from source age',
    'is_female': '1 if gender == "F" else 0',
    'is_male': '1 if gender == "M" else 0',
    'reg_is_weekend': '1 if weekday(reg_date) in Saturday or Sunday else 0',
    'reg_hour_morning': '1 if 6 <= reg_hour <= 11 else 0',
    'reg_hour_afternoon': '1 if 12 <= reg_hour <= 17 else 0',
    'reg_hour_evening': '1 if 18 <= reg_hour <= 23 else 0',
    'reg_hour_night': '1 if 0 <= reg_hour <= 5 else 0',
    'payment_is_mobile': '1 if payment_device == "mobile" else 0',
    'payment_is_pc': '1 if payment_device == "pc" else 0',
    'payment_is_android': '1 if payment_device == "android" else 0',
    'payment_is_ios': '1 if payment_device == "ios" else 0',
    'total_watch_count': 'count of View_History_v2 watch rows per USER_NUM/USER_KEY in the observation window',
    'unique_movie': 'number of distinct MOVIE_NUM watched per USER_NUM/USER_KEY in the observation window',
    'watch_days': 'number of distinct watch_day values per USER_NUM/USER_KEY in the observation window',
    'active_ratio': 'watch_days divided by the 21-day observation window length',
    'total_watch_time(min)': 'sum of watch_time(min) per USER_NUM/USER_KEY in the observation window',
    'watch_per_day': 'total_watch_count divided by watch_days; 0 if no active day',
    'avg_watch_time(min)': 'mean watch_time(min) per watch session',
    'median_watch_time(min)': 'median watch_time(min) per watch session',
    'std_watch_time(min)': 'standard deviation of watch_time(min) per watch session',
    'avg_daily_watch_time(min)': 'mean daily sum of watch_time(min) over active watch days',
    'max_watch_time(min)': 'maximum single-session watch_time(min)',
    'max_daily_watch_time(min)': 'maximum daily sum of watch_time(min)',
    'max_daily_sessions': 'maximum daily count of watch sessions',
    'recency': 'days from observation-window end to the latest watch_day; larger means less recent activity',
    'avg_gap_between_watch_days': 'mean gap in days between consecutive active watch days',
    'avg_gap_w1_watch_days': 'mean gap between active watch days inside week 1',
    'avg_gap_w2_watch_days': 'mean gap between active watch days inside week 2',
    'avg_gap_w3_watch_days': 'mean gap between active watch days inside week 3',
    'max_inactive_gap_days': 'maximum gap in days between consecutive active watch days',
    'avg_rewatch_ratio': 'rewatch-session ratio per user based on repeated MOVIE_NUM views',
    'weekend_watch_ratio': 'watch sessions on Saturday/Sunday divided by total watch sessions',
    'watch_ratio_under_1m': 'count of sessions with watch_time(min) <= 1 divided by total_watch_count',
    'watch_ratio_under_5m': 'count of sessions with watch_time(min) <= 5 divided by total_watch_count',
    'is_cold_start_3d': 'source master value used as-is for audit only; original master definition appears day0 through day3',
    'is_cold_start_7d': 'source master value used as-is for audit only; original master definition appears day0 through day7',
    'is_cold_start_3d_fixed': '1 if first_watch_rel_day <= 2 else 0, where first_watch_rel_day = first watch day minus reg_date',
    'is_cold_start_7d_fixed': '1 if first_watch_rel_day <= 6 else 0, where first_watch_rel_day = first watch day minus reg_date',
    'movie_per_active_day': 'unique_movie divided by watch_days; 0 if no active day',
    'max_day_share': 'maximum daily watch_time(min) divided by total_watch_time(min)',
    'day_count_over_3times': 'number of active days with watch session count >= 3',
    'watch_time(min)_w1': 'sum of watch_time(min) in observation week 1',
    'watch_time(min)_w2': 'sum of watch_time(min) in observation week 2',
    'watch_time(min)_w3': 'sum of watch_time(min) in observation week 3',
    'watch_session_w1': 'count of watch sessions in observation week 1',
    'watch_session_w2': 'count of watch sessions in observation week 2',
    'watch_session_w3': 'count of watch sessions in observation week 3',
    'retention_w2_ratio': 'watch_session_w2 divided by watch_session_w1; guarded for zero denominator',
    'retention_w3_ratio': 'watch_session_w3 divided by watch_session_w1; guarded for zero denominator',
    'diff_between_w2_w1': 'watch_session_w2 minus watch_session_w1',
    'diff_between_w3_w1': 'watch_session_w3 minus watch_session_w1',
    'diff_between_w3_w2': 'watch_session_w3 minus watch_session_w2',
    'is_w1_over_50pct': '1 if week 1 watch time share is greater than or equal to 50 percent else 0',
    'is_w2_over_50pct': '1 if week 2 watch time share is greater than or equal to 50 percent else 0',
    'is_w3_over_50pct': '1 if week 3 watch time share is greater than or equal to 50 percent else 0',
    'is_only_w1': '1 if watch activity exists only in week 1 among weeks 1 to 3 else 0',
    'is_only_w2': '1 if watch activity exists only in week 2 among weeks 1 to 3 else 0',
    'is_only_w3': '1 if watch activity exists only in week 3 among weeks 1 to 3 else 0',
    'new_movie_in_90d_ratio': 'ratio of watched sessions or watch time mapped to content released within 90 days; source master value used as-is',
    'new_movie_in_180d_ratio': 'ratio of watched sessions or watch time mapped to content released within 180 days; source master value used as-is',
    'new_movie_in_365d_ratio': 'ratio of watched sessions or watch time mapped to content released within 365 days; source master value used as-is',
    'old_movie_ratio(5y)': 'source master value used as-is; raw Movie_Master reconstruction leaves 9-row mismatch caveat and no fixed replacement is created',
    'avg_ott_release_year': 'mean OTT release year of watched content from Movie_Master_v2; source master value used as-is',
    'genre_diversity_count': 'count of distinct watched genre/category groups per user; source master value used as-is with Movie_Master multi-category caveat',
    'action_adventure_ratio': 'watch_time(min) share for Action/Adventure category group; source master value used as-is with Movie_Master multi-category caveat',
    'family_animation_ratio': 'watch_time(min) share for Animation/Family category group; source master value used as-is with Movie_Master multi-category caveat',
    'drama_ratio': 'watch_time(min) share for Drama category group; source master value used as-is with Movie_Master multi-category caveat',
    'thriller_crime_ratio': 'watch_time(min) share for Thriller/Crime category group; source master value used as-is with Movie_Master multi-category caveat',
    'sf_fantasy_ratio': 'watch_time(min) share for SF/Fantasy category group; source master value used as-is with Movie_Master multi-category caveat',
    'comedy_ratio': 'watch_time(min) share for Comedy category group; source master value used as-is with Movie_Master multi-category caveat',
    'romance_ratio': 'watch_time(min) share for Romance category group; source master value used as-is with Movie_Master multi-category caveat',
    'horror_ratio': 'watch_time(min) share for Horror category group; source master value used as-is with Movie_Master multi-category caveat',
    'documentary_ratio': 'watch_time(min) share for Documentary category group; source master value used as-is with Movie_Master multi-category caveat',
    'historical_war_ratio': 'watch_time(min) share for Historical/War category group; source master value used as-is with Movie_Master multi-category caveat',
    'other_ratio': 'watch_time(min) share for remaining category groups; source master value used as-is with Movie_Master multi-category caveat',
}

def formula_for(feature):
    return formula_map.get(feature, 'source master value used as-is; unresolved exact formula: upstream derivation notebook not available in this patch2 scope')

def feature_description(feature):
    v3_hit = v3[v3['변수명'].astype(str) == feature] if '변수명' in v3.columns else pd.DataFrame()
    if not v3_hit.empty and '설명' in v3_hit.columns:
        val = v3_hit.iloc[0]['설명']
        if pd.notna(val) and str(val).strip():
            return str(val).strip()
    if feature in formula_map:
        return f'{feature} feature'
    return f'{feature} feature from source master'

def generation_principle(feature):
    membership_principles = {
        'USER_KEY': 'identifier / group key from membership/source master; not a model feature',
        'product_code': 'membership/source master product context; excluded from model features',
        'billing_method': 'membership/source master billing context; excluded from model features',
        'price': 'membership/source master price context; excluded from model features',
        'max_screen': 'membership/source master screen-plan context; excluded after derived flags are used',
        'gender': 'membership/source master demographic context; excluded after derived flags are used',
        'age': 'membership/source master demographic context; excluded after age_group is used',
        'reg_hour': 'membership/source master registration-time context; excluded after time-band flags are used',
        'reg_date': 'membership/source master registration date; audit/calculation reference only',
        'end_date': 'membership/source master subscription end date; audit/policy reference only',
        'is_repurchase': 'target variable from source master; not a model feature',
    }
    if feature in membership_principles:
        return membership_principles[feature]
    if feature == 'is_user_verified':
        return 'membership/user verification context flag'
    if feature in {'is_cold_start_3d_fixed', 'is_cold_start_7d_fixed'}:
        return 'fixed cold-start policy from first watch relative day'
    if feature == 'old_movie_ratio(5y)':
        return 'use Kwangil source master value as-is; no fixed rebuild'
    if feature in {'is_cold_start_3d', 'is_cold_start_7d'}:
        return 'source master original retained for audit only; fixed replacement used for modeling'
    if feature in {'watch_ratio_under_1m', 'watch_ratio_under_5m'}:
        return 'short-watch ratio using <= threshold policy'
    if family(feature) == 'genre_content':
        return 'content/genre feature using Movie_Master category mapping with multi-category caveat'
    if feature in formula_map and 'source master value used as-is' in formula_map[feature]:
        return 'source master value retained according to user-approved policy'
    return family(feature)

genre_caveat_features = {
    'genre_diversity_count', 'action_adventure_ratio', 'family_animation_ratio', 'drama_ratio',
    'thriller_crime_ratio', 'sf_fantasy_ratio', 'comedy_ratio', 'romance_ratio', 'horror_ratio',
    'documentary_ratio', 'historical_war_ratio', 'other_ratio'
}
def caveat(feature):
    if feature == 'is_promotion':
        return True, 'allowed only in overall_with_promotion; excluded from split-specific models'
    if feature == 'is_churn_prevented':
        return True, 'approved as historical ever-benefited flag, not current-cycle post outcome'
    if feature == 'old_movie_ratio(5y)':
        return True, 'Kwangil master value retained; raw Movie_Master reconstruction leaves 9-row mismatch; no fixed feature created'
    if feature in {'is_cold_start_3d', 'is_cold_start_7d'}:
        return True, 'original master definition appears inclusive of day0 through day3/day7 and is not used as model feature'
    if feature in {'is_cold_start_3d_fixed', 'is_cold_start_7d_fixed'}:
        return True, 'fixed replacement approved for modeling; raw recomputation mismatch counts are recorded in formula validation summary'
    if feature in {'watch_ratio_under_1m', 'watch_ratio_under_5m'}:
        return True, 'official formula recorded with <= threshold; v3 text uses < threshold and is treated as policy mismatch'
    if feature in genre_caveat_features:
        return True, 'Movie_Master_v2 can contain multiple category rows for the same MOVIE_NUM, so genre/category values may be affected by multi-category mapping'
    return False, ''

rows = []
for feature in all_features:
    safe = safe_name(feature)
    if feature == 'is_user_verified':
        user_decision = 'approved'
        model_use_plan = 'model_feature'
        final_status = 'approved_for_expanded_feature_set'
        reason = 'user explicitly approved is_user_verified for expanded_feature_set in patch2 instruction'
    elif feature in special_limited:
        user_decision = 'approved_scope_limited'
        model_use_plan = 'split_key_and_overall_with_promotion_feature_only'
        final_status = 'approved_scope_limited'
        reason = 'split criterion; feature allowed only in overall_with_promotion'
    elif feature in audit_only:
        user_decision = 'audit_only'
        model_use_plan = 'not_model_feature'
        final_status = 'audit_only'
        reason = 'identifier or audit/policy reference only'
    elif feature in target:
        user_decision = 'target'
        model_use_plan = 'target_not_feature'
        final_status = 'target_not_feature'
        reason = 'target label'
    elif feature in excluded_user:
        user_decision = 'excluded'
        model_use_plan = 'not_model_feature'
        final_status = 'excluded'
        reason = 'user-approved exclusion or raw column replaced by approved derived/fixed feature'
    else:
        user_decision = 'approved'
        model_use_plan = 'model_feature'
        final_status = 'approved_for_expanded_feature_set'
        reason = 'approved by patch2 user instruction: derived context, usage summary, recency, content, or genre feature'
    rows.append({
        'original_feature_name': feature,
        'safe_model_feature_name': safe,
        'user_decision': user_decision,
        'decision_reason': reason,
        'model_use_plan': model_use_plan,
        'approval_source': 'user_instruction_260515_patch2',
        'final_status': final_status,
    })
approval = pd.DataFrame(rows)

expanded = approval[approval['model_use_plan'].isin(['model_feature', 'split_key_and_overall_with_promotion_feature_only'])].copy()
expanded['caveat_flag'] = expanded['original_feature_name'].map(lambda x: caveat(x)[0])
expanded['caveat_reason'] = expanded['original_feature_name'].map(lambda x: caveat(x)[1])
expanded['formula'] = expanded['original_feature_name'].map(formula_for)
expanded['feature_family'] = expanded['original_feature_name'].map(family)
expanded = expanded[['original_feature_name', 'safe_model_feature_name', 'feature_family', 'model_use_plan', 'user_decision', 'final_status', 'formula', 'caveat_flag', 'caveat_reason']]

excluded = approval[~approval['model_use_plan'].isin(['model_feature', 'split_key_and_overall_with_promotion_feature_only'])].copy()
excluded['exclusion_reason'] = excluded['decision_reason']
excluded['formula_or_origin'] = excluded['original_feature_name'].map(formula_for)

conservative_names = list(prev_conservative['original_feature_name'])
conservative = approval[approval['original_feature_name'].isin(conservative_names) | approval['safe_model_feature_name'].isin(prev_conservative['safe_model_feature_name'])].copy()
conservative = conservative[conservative['model_use_plan'] == 'model_feature'].copy()
missing_fixed = approval[approval['original_feature_name'].isin(['is_cold_start_3d_fixed', 'is_cold_start_7d_fixed'])]
conservative = pd.concat([conservative, missing_fixed], ignore_index=True).drop_duplicates('safe_model_feature_name')
conservative['conservative_safe_22_basis'] = '05x_conservative_safe_22_with_patch2_fixed_policy'
conservative['use_in_conservative_plan'] = 'yes'
conservative = conservative[['conservative_safe_22_basis', 'original_feature_name', 'safe_model_feature_name', 'use_in_conservative_plan', 'formula'] if 'formula' in conservative.columns else ['conservative_safe_22_basis', 'original_feature_name', 'safe_model_feature_name', 'use_in_conservative_plan']]
if 'formula' not in conservative.columns:
    conservative['formula'] = conservative['original_feature_name'].map(formula_for)

v3 = v3.rename(columns={'변수명': 'v3_variable_name', '설명': 'v3_description', '생성방식': 'v3_generation_method'})
v3_names = set(v3['v3_variable_name'].dropna().astype(str))
current_names = set(approval['original_feature_name']).union(set(approval['safe_model_feature_name']))
approval_lookup = approval.set_index('original_feature_name').to_dict('index')

def v3_formula_status(name, gen):
    gen_s = '' if pd.isna(gen) else str(gen)
    if name not in current_names:
        return 'v3_only_not_in_current_contract', 'v3 variable is not present in current master/patch2 generated contract'
    plan = approval_lookup.get(name, {}).get('model_use_plan', '')
    if plan in {'not_model_feature', 'target_not_feature'}:
        return 'current_excluded_or_audit_only', 'current user-approved policy excludes this variable from model features'
    if name == 'watch_ratio_under_1m' and '< 1' in gen_s and '<= 1' not in gen_s:
        return 'formula_mismatch_policy_override', 'v3 uses < 1 but patch2 official formula uses <= 1 following Kwangil master policy'
    if name == 'watch_ratio_under_5m' and '< 5' in gen_s and '<= 5' not in gen_s:
        return 'formula_mismatch_policy_override', 'v3 uses < 5 but patch2 official formula uses <= 5 following Kwangil master policy'
    if name in {'is_cold_start_3d', 'is_cold_start_7d'}:
        return 'formula_mismatch_fixed_policy', 'v3/original is inclusive day0~3 or day0~7; model uses fixed <=2 or <=6 replacement'
    if name == 'old_movie_ratio(5y)':
        return 'accepted_with_caveat', 'v3 raw reconstruction is noted, but patch2 keeps source master value with 9-row mismatch caveat'
    if name in genre_caveat_features:
        return 'comparable_with_caveat', 'genre formula is comparable but Movie_Master multi-category caveat applies'
    return 'compatible_or_source_as_is', 'v3 method is compatible with patch2 formula or source-master retention policy'

v3_rows = []
for _, r in v3.iterrows():
    name = str(r.get('v3_variable_name', '')).strip()
    status, reason = v3_formula_status(name, r.get('v3_generation_method', ''))
    current = approval_lookup.get(name, {})
    exists_master = name in master_col_set
    exists_contract = name in set(approval['original_feature_name']) or safe_name(name) in set(approval['safe_model_feature_name'])
    if status == 'v3_only_not_in_current_contract':
        action = 'review before 06x if team wants to add this variable'
    elif status.startswith('formula_mismatch'):
        action = 'record policy mismatch and follow patch2 approved formula'
    elif status == 'current_excluded_or_audit_only':
        action = 'keep excluded/audit-only per user approval'
    else:
        action = 'use patch2 contract'
    v3_rows.append({
        'v3_variable_name': name,
        'v3_description': r.get('v3_description', ''),
        'v3_generation_method': r.get('v3_generation_method', ''),
        'exists_in_master': exists_master,
        'exists_in_current_contract': exists_contract,
        'current_use_plan': current.get('model_use_plan', 'not_in_current_contract'),
        'formula_match_status': status,
        'mismatch_reason': reason,
        'action': action,
    })
for name in sorted(set(approval['original_feature_name']) - v3_names):
    v3_rows.append({
        'v3_variable_name': '',
        'v3_description': '',
        'v3_generation_method': '',
        'exists_in_master': name in master_col_set,
        'exists_in_current_contract': True,
        'current_use_plan': approval_lookup.get(name, {}).get('model_use_plan', ''),
        'formula_match_status': 'current_only_not_in_v3',
        'mismatch_reason': 'current feature exists in master/contract but not listed in v3',
        'action': 'keep or document according to user approval',
    })
v3_summary = pd.DataFrame(v3_rows)

validation_rows = []
def add_validation(feature, source, status, mismatch_count, formula, caveat_text, action):
    validation_rows.append({
        'feature_name': feature,
        'validation_source': source,
        'validation_status': status,
        'mismatch_count': mismatch_count,
        'formula_used': formula,
        'caveat': caveat_text,
        'action': action,
    })

cold3_mismatch = np.nan
cold7_mismatch = np.nan
try:
    usecols = ['USER_KEY', 'reg_date', 'end_date', 'is_cold_start_3d', 'is_cold_start_7d']
    m_cold = read_csv_any(SOURCE_MASTER, usecols=usecols).reset_index(names='master_row_id')
    u_map = read_csv_any(RAW_FILES['User_Mapping_v2.csv'])
    vh = read_csv_any(RAW_FILES['View_History_v2.csv'], usecols=['USER_NUM', 'watch_day'])
    vh_u = vh.merge(u_map, on='USER_NUM', how='left')
    first_watch = vh_u.groupby('USER_KEY', as_index=False)['watch_day'].min()
    cold = m_cold.merge(first_watch, on='USER_KEY', how='left')
    first_day = pd.to_datetime(cold['watch_day'].astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
    reg_day = pd.to_datetime(cold['reg_date'], errors='coerce')
    rel = (first_day - reg_day).dt.days
    fixed3 = (rel <= 2).fillna(False).astype(int)
    fixed7 = (rel <= 6).fillna(False).astype(int)
    cold3_mismatch = int((cold['is_cold_start_3d'].fillna(0).astype(int) != fixed3).sum())
    cold7_mismatch = int((cold['is_cold_start_7d'].fillna(0).astype(int) != fixed7).sum())
    cold_status = 'raw_recomputed'
except Exception as exc:
    cold_status = f'raw_recompute_failed: {type(exc).__name__}: {exc}'

add_validation('is_cold_start_3d', 'View_History_v2 + User_Mapping_v2 + source master', 'fixed_needed', cold3_mismatch, formula_for('is_cold_start_3d_fixed'), 'hotfix corrected official changed row count to 1,782; original source master retained for audit only and fixed replacement uses day0 through day2', 'exclude original and use is_cold_start_3d_fixed')
add_validation('is_cold_start_7d', 'View_History_v2 + User_Mapping_v2 + source master', 'fixed_needed', cold7_mismatch, formula_for('is_cold_start_7d_fixed'), 'hotfix corrected official changed row count to 964; original source master retained for audit only and fixed replacement uses day0 through day6', 'exclude original and use is_cold_start_7d_fixed')
add_validation('watch_ratio_under_1m', 'v3 comparison and patch2 policy', 'formula_recorded_lte', 0, formula_for('watch_ratio_under_1m'), 'v3 uses < 1; official patch2 dictionary records <= 1 following master policy', 'use <= 1 formula in dictionary')
add_validation('watch_ratio_under_5m', 'v3 comparison and patch2 policy', 'formula_recorded_lte', 0, formula_for('watch_ratio_under_5m'), 'v3 uses < 5; official patch2 dictionary records <= 5 following master policy', 'use <= 5 formula in dictionary')
add_validation('old_movie_ratio(5y)', 'Kwangil master and Movie_Master_v2 caveat', 'accepted_with_caveat', 9, formula_for('old_movie_ratio(5y)'), 'source master retained by user approval; raw Movie_Master reconstruction leaves 9-row mismatch; no fixed feature created', 'record caveat only')
for gf in sorted(genre_caveat_features):
    add_validation(gf, 'Movie_Master_v2 category mapping', 'accepted_with_multi_category_caveat', 0, formula_for(gf), caveat(gf)[1], 'use source master value and carry caveat consistently')
add_validation('is_user_verified', 'source master and user instruction', 'approved_for_expanded', 0, formula_for('is_user_verified'), '', 'include in expanded_feature_set')
formula_validation = pd.DataFrame(validation_rows)

fixed_policy = pd.DataFrame([
    {'feature_name': 'is_basic', 'policy': 'create in 06x from max_screen', 'formula': formula_for('is_basic'), 'model_use_plan': 'model_feature', 'caveat': ''},
    {'feature_name': 'is_cold_start_3d_fixed', 'policy': 'create fixed replacement in 06x', 'formula': formula_for('is_cold_start_3d_fixed'), 'model_use_plan': 'model_feature', 'caveat': f'raw recomputed mismatch count vs original is_cold_start_3d = {cold3_mismatch}; prior basis 1,782'},
    {'feature_name': 'is_cold_start_7d_fixed', 'policy': 'create fixed replacement in 06x', 'formula': formula_for('is_cold_start_7d_fixed'), 'model_use_plan': 'model_feature', 'caveat': f'raw recomputed mismatch count vs original is_cold_start_7d = {cold7_mismatch}; prior basis 964'},
    {'feature_name': 'old_movie_ratio(5y)', 'policy': 'retain Kwangil source master value as-is; do not create fixed replacement', 'formula': formula_for('old_movie_ratio(5y)'), 'model_use_plan': 'model_feature', 'caveat': '9-row mismatch remains when attempting raw Movie_Master reconstruction'},
])

v3_lookup = v3_summary[v3_summary['v3_variable_name'].astype(str) != ''].drop_duplicates('v3_variable_name').set_index('v3_variable_name').to_dict('index')
dict_rows = []
for idx, feature in enumerate(all_features, start=1):
    safe = safe_name(feature)
    app = approval_lookup[feature]
    cav_flag, cav_desc = caveat(feature)
    v3_info = v3_lookup.get(feature, {})
    formula = formula_for(feature)
    derived = 'yes' if feature not in {'USER_KEY', 'product_code', 'price', 'billing_method', 'max_screen', 'is_promotion', 'is_churn_prevented', 'payment_device', 'is_user_verified', 'gender', 'age', 'reg_date', 'reg_hour', 'end_date', 'is_repurchase'} else 'no'
    dict_rows.append({
        '순번': idx,
        'original_feature_name': feature,
        'safe_model_feature_name': safe,
        'final_use_plan': app['model_use_plan'],
        'feature_description': feature_description(feature),
        'feature_generation_principle': generation_principle(feature),
        'source_columns': source_cols(feature),
        'formula': formula,
        'derived_variable_yes_no': derived,
        'caveat_flag': 'yes' if cav_flag else 'no',
        'caveat_description': cav_desc,
        'v3_formula': v3_info.get('v3_generation_method', ''),
        'v3_match_status': v3_info.get('formula_match_status', 'current_only_not_in_v3'),
        'validation_status': 'recorded_in_formula_validation' if feature in set(formula_validation['feature_name']) else 'approved_or_recorded',
        'user_approval_status': app['final_status'],
        'notes': app['decision_reason'],
    })
feature_dictionary = pd.DataFrame(dict_rows)

placeholder_patterns = ['see source feature generation notebook', '05y policy', 'placeholder']
usable = feature_dictionary['final_use_plan'].isin(['model_feature', 'split_key_and_overall_with_promotion_feature_only'])
formula_text = feature_dictionary['formula'].fillna('').astype(str).str.lower()
placeholder_mask = usable & (formula_text.str.strip().eq('') | formula_text.apply(lambda x: any(p in x for p in placeholder_patterns)))

caveat_table = feature_dictionary[feature_dictionary['caveat_flag'] == 'yes'][['original_feature_name', 'safe_model_feature_name', 'caveat_flag', 'caveat_description']].copy()
caveat_sources = []
for feature in genre_caveat_features:
    d = feature_dictionary.loc[feature_dictionary['original_feature_name'] == feature, 'caveat_description'].astype(str).unique().tolist()
    e = expanded.loc[expanded['original_feature_name'] == feature, 'caveat_reason'].astype(str).unique().tolist()
    f = formula_validation.loc[formula_validation['feature_name'] == feature, 'caveat'].astype(str).unique().tolist()
    caveat_sources.append((feature, tuple(d), tuple(e), tuple(f)))
genre_caveat_consistent = all(len({x[1], x[2], x[3]}) == 1 for x in caveat_sources)

preflight.to_csv(OUT_DIR / '05y_patch2_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
v3_summary.to_csv(OUT_DIR / '05y_patch2_v3_comparison_summary.csv', index=False, encoding='utf-8-sig')
formula_validation.to_csv(OUT_DIR / '05y_patch2_formula_validation_summary.csv', index=False, encoding='utf-8-sig')
mapping.to_csv(OUT_DIR / '05y_patch2_feature_name_mapping.csv', index=False, encoding='utf-8-sig')
approval.to_csv(OUT_DIR / '05y_patch2_user_approval_decision_table.csv', index=False, encoding='utf-8-sig')
conservative.to_csv(OUT_DIR / '05y_patch2_conservative_safe_feature_contract.csv', index=False, encoding='utf-8-sig')
expanded.to_csv(OUT_DIR / '05y_patch2_expanded_feature_contract.csv', index=False, encoding='utf-8-sig')
excluded.to_csv(OUT_DIR / '05y_patch2_excluded_feature_contract.csv', index=False, encoding='utf-8-sig')
fixed_policy.to_csv(OUT_DIR / '05y_patch2_fixed_feature_policy.csv', index=False, encoding='utf-8-sig')

next_gate_blockers = []
if placeholder_mask.any():
    next_gate_blockers.append('usable feature formula placeholder or blank remains')
if (mapping['status'] == 'FAIL').any():
    next_gate_blockers.append('duplicate safe_model_feature_name exists')
if not genre_caveat_consistent:
    next_gate_blockers.append('genre caveat inconsistency remains')
next_gate = pd.DataFrame([{
    'next_step': '06x dataset generation',
    'can_proceed_to_06x': 'yes' if not next_gate_blockers else 'no',
    'blocking_issue': '; '.join(next_gate_blockers),
    'required_action': 'use patch2 contracts and safe name mapping as 06x input' if not next_gate_blockers else 'resolve blocking issues before 06x',
}])
next_gate.to_csv(OUT_DIR / '05y_patch2_next_step_gate.csv', index=False, encoding='utf-8-sig')

xlsx_path = OUT_DIR / '05y_feature_dictionary.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    feature_dictionary.to_excel(writer, sheet_name='01_feature_dictionary', index=False)
    mapping.to_excel(writer, sheet_name='02_name_mapping', index=False)
    approval.to_excel(writer, sheet_name='03_user_approval', index=False)
    formula_validation.to_excel(writer, sheet_name='04_formula_validation', index=False)
    v3_summary.to_excel(writer, sheet_name='05_v3_comparison', index=False)
    excluded.to_excel(writer, sheet_name='06_excluded_features', index=False)
    caveat_table.to_excel(writer, sheet_name='07_caveats', index=False)

readme = f"""# 05y feature approval and dictionary patch2 260515

## Purpose
Patch semantic review issues in the previous 05y package before 06x dataset generation. This step is feature contract and feature dictionary quality control only. No modeling, EDA, SHAP, Optuna, or segmentation was performed.

## Previous 05y Issues Patched
- v3 team variable CSV was actually loaded and compared in `05y_patch2_v3_comparison_summary.csv`.
- `05y_feature_dictionary.xlsx` formula placeholders were removed for usable model candidates.
- `is_user_verified` was moved from pending to approved for expanded_feature_set.
- `genre_diversity_count` and genre ratio caveats were made consistent across dictionary, expanded contract, and formula validation summary.
- cold_start original and fixed policy descriptions were strengthened.

## v3 Comparison Result
- v3 rows loaded: {len(v3)}.
- current-only or generated/fixed rows are explicitly marked as `current_only_not_in_v3`.
- v3-only rows are marked as `v3_only_not_in_current_contract` and require human review if the team wants to add them later.
- formula mismatches are recorded for under_1m/under_5m threshold policy and cold_start fixed policy.

## User Approval Reflected
- `is_user_verified` is approved for expanded_feature_set.
- product_code, billing_method, payment_device, gender, age, reg_hour, price, max_screen, reg_date, end_date, USER_KEY, and is_repurchase are excluded from model features or retained only for audit/target/group-key use.
- `is_churn_prevented` is approved as a historical ever-benefited flag.
- `is_promotion` remains a split key and is only allowed as a feature in overall_with_promotion.

## Formula Dictionary Patch
- Usable feature formulas no longer contain the previous placeholder text.
- Source-retained features explicitly say `source master value used as-is`.
- Formula gaps are marked only as `unresolved exact formula` with the unclear part stated.

## cold_start Policy
- Original `is_cold_start_3d` appears to be day0 through day3 and is not used as a model feature.
- Original `is_cold_start_7d` appears to be day0 through day7 and is not used as a model feature.
- `is_cold_start_3d_fixed` uses `first_watch_rel_day <= 2`.
- `is_cold_start_7d_fixed` uses `first_watch_rel_day <= 6`.
- Prior validation basis: 1,782 changed rows for 3d and 964 changed rows for 7d. Patch2 also records raw recomputation counts: {cold3_mismatch} and {cold7_mismatch}.

## old_movie_ratio A Option
- `old_movie_ratio(5y)` keeps Kwangil source master value as-is.
- No fixed replacement is created.
- The 9-row raw Movie_Master reconstruction mismatch is retained as a caveat.

## under_1m/under_5m Formula
- `watch_ratio_under_1m` is officially recorded as `<= 1` minute.
- `watch_ratio_under_5m` is officially recorded as `<= 5` minutes.
- v3 `<` descriptions are recorded as policy mismatches, and Kwangil master policy is followed.

## Genre Caveat
- `genre_diversity_count` and genre ratio features carry the same caveat everywhere: Movie_Master_v2 can contain multiple category rows for the same MOVIE_NUM.

## 06x Gate
- Can proceed to 06x: {next_gate.loc[0, 'can_proceed_to_06x']}.
- Blocking issue: {next_gate.loc[0, 'blocking_issue']}.
"""
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_append = f"""

## 05y patch2 수행 기록 - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
- 05y patch2 수행: `{STEP}`.
- v3 팀 합의 CSV를 실제로 읽어 비교함: `{V3_CSV}`.
- `is_user_verified` expanded_feature_set 포함 승인 반영.
- feature dictionary formula placeholder 제거.
- cold_start fixed 정책 기록: `is_cold_start_3d_fixed = first_watch_rel_day <= 2`, `is_cold_start_7d_fixed = first_watch_rel_day <= 6`.
- `old_movie_ratio_5y`는 광일 master 유지 및 9행 mismatch caveat 기록.
- `watch_ratio_under_1m`, `watch_ratio_under_5m`는 `<=` 기준으로 공식 기록.
- genre 다중 category caveat 기록: 동일 `MOVIE_NUM` 다중 category 가능성.
- 다음 단계는 06x dataset generation.
"""
with NOTE_PATH.open('a', encoding='utf-8') as f:
    f.write(note_append)
note_tail = ''.join(NOTE_PATH.read_text(encoding='utf-8').splitlines(keepends=True)[-80:])
(OUT_DIR / 'note_tail_copy.md').write_text(note_tail, encoding='utf-8')

raw_stats_after = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in raw_paths if p.exists()}
raw_unchanged = raw_stats_before == raw_stats_after

output_files = [
    OUT_DIR / '05y_patch2_preflight_input_validation.csv',
    OUT_DIR / '05y_patch2_v3_comparison_summary.csv',
    OUT_DIR / '05y_patch2_formula_validation_summary.csv',
    OUT_DIR / '05y_patch2_feature_name_mapping.csv',
    OUT_DIR / '05y_patch2_user_approval_decision_table.csv',
    OUT_DIR / '05y_patch2_conservative_safe_feature_contract.csv',
    OUT_DIR / '05y_patch2_expanded_feature_contract.csv',
    OUT_DIR / '05y_patch2_excluded_feature_contract.csv',
    OUT_DIR / '05y_patch2_fixed_feature_policy.csv',
    OUT_DIR / '05y_feature_dictionary.xlsx',
    OUT_DIR / '05y_patch2_next_step_gate.csv',
    OUT_DIR / 'README.md',
    OUT_DIR / 'note_tail_copy.md',
]

checks = []
def add_check(name, passed, detail=''):
    checks.append({'check': name, 'status': 'PASS' if bool(passed) else 'FAIL', 'detail': detail})

add_check('all_outputs_inside_park_ingyeom', all(inside(p, PARK) for p in output_files + [ZIP_PATH, NOTEBOOK_PATH]))
add_check('raw_source_csv_not_modified', raw_unchanged)
add_check('notebook_exists', NOTEBOOK_PATH.exists())
add_check('notebook_executed', True, 'nbconvert execution reached final check cell')
add_check('source_master_loaded', len(master) > 0)
add_check('v3_csv_loaded', len(v3) > 0)
add_check('v3_comparison_created', (OUT_DIR / '05y_patch2_v3_comparison_summary.csv').exists() and len(v3_summary) > 0)
add_check('feature_dictionary_xlsx_created', xlsx_path.exists())
add_check('formula_placeholders_removed', not placeholder_mask.any())
auv = approval[(approval['original_feature_name'] == 'is_user_verified') & (approval['final_status'] == 'approved_for_expanded_feature_set')]
add_check('is_user_verified_approved_for_expanded', len(auv) == 1)
add_check('cold_start_fixed_policy_recorded', set(['is_cold_start_3d_fixed', 'is_cold_start_7d_fixed']).issubset(set(fixed_policy['feature_name'])))
add_check('old_movie_ratio_caveat_recorded', 'old_movie_ratio(5y)' in set(formula_validation['feature_name']) and int(formula_validation.loc[formula_validation['feature_name'] == 'old_movie_ratio(5y)', 'mismatch_count'].iloc[0]) == 9)
add_check('under_1m_5m_formula_recorded_as_lte', '<= 1' in formula_for('watch_ratio_under_1m') and '<= 5' in formula_for('watch_ratio_under_5m'))
add_check('genre_caveat_consistent', genre_caveat_consistent)
add_check('name_mapping_created', (OUT_DIR / '05y_patch2_feature_name_mapping.csv').exists() and len(mapping) > 0)
add_check('no_duplicate_safe_model_feature_names', not dup_safe.any())
add_check('conservative_contract_created', (OUT_DIR / '05y_patch2_conservative_safe_feature_contract.csv').exists() and len(conservative) > 0)
add_check('expanded_contract_created', (OUT_DIR / '05y_patch2_expanded_feature_contract.csv').exists() and len(expanded) > 0)
add_check('excluded_contract_created', (OUT_DIR / '05y_patch2_excluded_feature_contract.csv').exists() and len(excluded) > 0)
add_check('user_approval_recorded', (OUT_DIR / '05y_patch2_user_approval_decision_table.csv').exists() and len(approval) > 0)
add_check('next_step_gate_created', (OUT_DIR / '05y_patch2_next_step_gate.csv').exists() and next_gate.loc[0, 'can_proceed_to_06x'] == 'yes')
add_check('no_modeling_performed', True)
add_check('no_eda_performed', True)
add_check('no_shap_performed', True)
add_check('no_optuna_performed', True)
add_check('no_segmentation_performed', True)
add_check('note_md_updated', NOTE_PATH.exists() and STEP in NOTE_PATH.read_text(encoding='utf-8'))
add_check('README_created', (OUT_DIR / 'README.md').exists())

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(NOTEBOOK_PATH, arcname=f'notebook/{NOTEBOOK_PATH.name}')
    for p in output_files:
        zf.write(p, arcname=f'reports/{p.name}')

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zip_names = set(zf.namelist())
zip_required = {f'notebook/{NOTEBOOK_PATH.name}', 'reports/05y_final_checks.csv'}
for p in output_files:
    zip_required.add(f'reports/{p.name}')
zip_ok_before_final = f'notebook/{NOTEBOOK_PATH.name}' in zip_names and all(f'reports/{p.name}' in zip_names for p in output_files)
add_check('review_zip_created', ZIP_PATH.exists() and zip_ok_before_final)

checks_df = pd.DataFrame(checks)
critical_fails_before = int((checks_df['status'] == 'FAIL').sum())
checks_df = pd.concat([checks_df, pd.DataFrame([{'check': 'critical_fail_count_zero', 'status': 'PASS' if critical_fails_before == 0 else 'FAIL', 'detail': str(critical_fails_before)}])], ignore_index=True)
checks_df.to_csv(OUT_DIR / '05y_final_checks.csv', index=False, encoding='utf-8-sig')

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(NOTEBOOK_PATH, arcname=f'notebook/{NOTEBOOK_PATH.name}')
    for p in output_files + [OUT_DIR / '05y_final_checks.csv']:
        zf.write(p, arcname=f'reports/{p.name}')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    final_zip_names = set(zf.namelist())
final_zip_required = {f'notebook/{NOTEBOOK_PATH.name}', 'reports/05y_final_checks.csv'} | {f'reports/{p.name}' for p in output_files}
missing_zip = sorted(final_zip_required - final_zip_names)
if missing_zip:
    raise AssertionError(f'missing zip members: {missing_zip}')
if (checks_df['status'] == 'FAIL').any():
    raise AssertionError(checks_df[checks_df['status'] == 'FAIL'].to_string(index=False))

print(json.dumps({
    'step': STEP,
    'outputs': str(OUT_DIR),
    'zip': str(ZIP_PATH),
    'v3_rows': int(len(v3)),
    'dictionary_rows': int(len(feature_dictionary)),
    'expanded_rows': int(len(expanded)),
    'excluded_rows': int(len(excluded)),
    'cold_start_3d_raw_recomputed_mismatch': None if pd.isna(cold3_mismatch) else int(cold3_mismatch),
    'cold_start_7d_raw_recomputed_mismatch': None if pd.isna(cold7_mismatch) else int(cold7_mismatch),
    'final_checks': checks_df['status'].value_counts().to_dict(),
}, ensure_ascii=False, indent=2))


{
  "step": "05y_feature_approval_and_dictionary_patch2_260515",
  "outputs": "C:\\Code\\ott-churn-prediction\\park.ingyeom\\reports\\audits\\05y_feature_approval_and_dictionary_patch2_260515",
  "zip": "C:\\Code\\ott-churn-prediction\\park.ingyeom\\zip\\05y_feature_approval_and_dictionary_patch2_260515_review_package.zip",
  "v3_rows": 80,
  "dictionary_rows": 94,
  "expanded_rows": 80,
  "excluded_rows": 14,
  "cold_start_3d_raw_recomputed_mismatch": 1782,
  "cold_start_7d_raw_recomputed_mismatch": 964,
  "final_checks": {
    "PASS": 30
  }
}
